In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

In [3]:
df = pd.read_csv("Hotel_Reviews.csv")
print("Shape:", df.shape)
print(df[["Positive_Review", "Negative_Review", "Reviewer_Score"]].head())

Shape: (26844, 17)
                                     Positive_Review  \
0   Only the park outside of the hotel was beauti...   
1   No real complaints the hotel was great great ...   
2   Location was good and staff were ok It is cut...   
3   Great location in nice surroundings the bar a...   
4    Amazing location and building Romantic setting    

                                     Negative_Review  Reviewer_Score  
0   I am so angry that i made this post available...             2.9  
1                                        No Negative             7.5  
2   Rooms are nice but for elderly a bit difficul...             7.1  
3   My room was dirty and I was afraid to walk ba...             3.8  
4   You When I booked with your company on line y...             6.7  


In [10]:
# ------------------------------------------------------------
# 2. Build a labelled text dataset
#    label 1 = positive, 0 = negative
# ------------------------------------------------------------
print("Rows in df:", len(df))

# Use up to 100,000 rows (works even if your file is smaller)
n = min(100_000, len(df))
df = df.sample(n, random_state=42).reset_index(drop=True)

pos = df[["Positive_Review"]].rename(columns={"Positive_Review": "text"})
pos["label"] = 1
neg = df[["Negative_Review"]].rename(columns={"Negative_Review": "text"})
neg["label"] = 0

data = pd.concat([pos, neg], ignore_index=True)

# Remove placeholder / empty reviews
placeholders = ["No Negative", "No Positive", "Nothing", "nothing", "N A", "None"]
data["text"] = data["text"].astype(str).str.strip()
data = data[~data["text"].isin(placeholders)]
data = data[data["text"].str.len() > 3]
data = data.drop_duplicates(subset="text").reset_index(drop=True)

print("\nClass balance:")
print(data["label"].value_counts())

Rows in df: 26844

Class balance:
label
1    22639
0    17818
Name: count, dtype: int64


In [22]:
# ------------------------------------------------------------
# 3. Clean text (light cleaning; TF-IDF handles the rest)
# ------------------------------------------------------------
data["text"] = (
    data["text"]
    .str.lower()
    .str.replace(r"n't\b", " not", regex=True)
    .str.replace(r"[^a-z\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [13]:
# ------------------------------------------------------------
# 4. Train / test split
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    data["text"], data["label"],
    test_size=0.2, random_state=42, stratify=data["label"],
)

In [23]:
# ------------------------------------------------------------
# 5. TF-IDF features
# ------------------------------------------------------------
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = list(ENGLISH_STOP_WORDS - {"not", "no", "never", "nor", "nothing", "cannot", "without"})

tfidf = TfidfVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),
    max_features=50_000,
    min_df=5,
)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

In [24]:
# ------------------------------------------------------------
# 6. Model: Logistic Regression (strong baseline for text)
# ------------------------------------------------------------
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_vec, y_train)


LogisticRegression(class_weight='balanced', max_iter=1000)

In [25]:
# ------------------------------------------------------------
# 7. Evaluate
# ------------------------------------------------------------
preds = model.predict(X_test_vec)
print("\nClassification report:")
print(classification_report(y_test, preds, target_names=["Negative", "Positive"]))
print("Macro F1:", round(f1_score(y_test, preds, average="macro"), 4))

cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.close()


Classification report:
              precision    recall  f1-score   support

    Negative       0.91      0.95      0.93      3564
    Positive       0.96      0.93      0.94      4528

    accuracy                           0.94      8092
   macro avg       0.93      0.94      0.94      8092
weighted avg       0.94      0.94      0.94      8092

Macro F1: 0.9357


In [26]:
# ------------------------------------------------------------
# 8. Interpret: which words drive each sentiment?
# ------------------------------------------------------------
feature_names = np.array(tfidf.get_feature_names_out())
coefs = model.coef_[0]
top_pos = feature_names[np.argsort(coefs)[-15:]][::-1]
top_neg = feature_names[np.argsort(coefs)[:15]]
print("\nTop positive words:", list(top_pos))
print("Top negative words:", list(top_neg))


Top positive words: ['location', 'comfortable', 'great', 'excellent', 'helpful', 'staff', 'comfy', 'lovely', 'clean', 'nice', 'beautiful', 'friendly', 'quiet', 'amazing', 'good']
Top negative words: ['not', 'no', 'bit', 'small', 'wasn', 'poor', 'noisy', 'nothing', 'little', 'expensive', 'slow', 'didn', 'rude', 'uncomfortable', 'difficult']


In [18]:
# ------------------------------------------------------------
# 9. Save model and predict on new text
# ------------------------------------------------------------
joblib.dump(model, "sentiment_model.joblib")
joblib.dump(tfidf, "tfidf_vectorizer.joblib")


def predict_sentiment(text: str) -> str:
    cleaned = pd.Series([text]).str.lower().str.replace(r"[^a-z\s]", " ", regex=True)
    vec = tfidf.transform(cleaned)
    prob = model.predict_proba(vec)[0, 1]
    label = "Positive" if prob >= 0.5 else "Negative"
    return f"{label} ({prob:.2f} positive probability)"


print("\nDemo predictions:")
print(predict_sentiment("Amazing location and very friendly staff"))
print(predict_sentiment("The room was dirty and the staff were rude"))


Demo predictions:
Positive (1.00 positive probability)
Negative (0.04 positive probability)


In [20]:
import ipywidgets as widgets
from IPython.display import display

text_box = widgets.Textarea(
    placeholder="Type or paste a hotel review here...",
    layout=widgets.Layout(width="600px", height="100px"),
)
button = widgets.Button(description="Analyze Review", button_style="primary")
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        review = text_box.value.strip()
        if not review:
            print("Please type a review first.")
            return
        cleaned = pd.Series([review]).str.lower().str.replace(r"[^a-z\s]", " ", regex=True)
        prob = model.predict_proba(tfidf.transform(cleaned))[0, 1]
        label = "Positive" if prob >= 0.5 else "Negative"
        print(f"Sentiment: {label}")
        print(f"Positive probability: {prob:.2f}")

button.on_click(on_click)
display(text_box, button, output)

Textarea(value='', layout=Layout(height='100px', width='600px'), placeholder='Type or paste a hotel review her…

Button(button_style='primary', description='Analyze Review', style=ButtonStyle())

Output()